<a href="https://colab.research.google.com/github/fmipython/PythonCourse2022/blob/dsa/05%20-%20Data%20Structures%20and%20Algorithms/05%20-%20Data%20Structures%20and%20Algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Structures & Oddities

План на лекцията:

1. Преговор на познатите колекции и още операции с низове
2. Сложност на операциите: време и памет
3. Какво стои зад `list`, `dict` и `set`
4. Копиране на вложени колекции
5. `defaultdict`, `Counter` и `frozenset`
6. Стек, опашка с `deque` и пример за обхождане
7. Приоритетни опашки: `heapq` и `PriorityQueue`
8. Таблица: коя структура за каква задача?
9. Задача: изборът на структура има значение

Накрая има **допълнителни материали и справки**. Те остават в тази тетрадка и могат да се разглеждат отделно от основната последователност.

## Да си припомним наученото досега 🐍

Нека започнем с познатите ни колекции и операциите, които вече използвахме върху тях.

### `list`

Списъкът пази елементите в ред, допуска повторения и може да се променя (mutable). Достъпваме елементите по индекс, а със slicing взимаме част от списъка:

In [1]:
students = ["Иван", "Мария", "Иван"]
students.append("Петър")
students[0] = "Ани"

print(students)
print(students[0], students[-1])
print(students[1:3])

['Ани', 'Мария', 'Иван', 'Петър']
Ани Петър
['Мария', 'Иван']


### `tuple`

Кортежът също пази елементите в ред, но не можем да добавяме, премахваме или заменяме елементите му (immutable). Удобен е, когато искаме да съберем няколко стойности — например координатите на точка.

⚠️ За `tuple` с един елемент ни трябва запетайка: `(42,)`.

In [2]:
point = (2, 3)
x, y = point

print(point[0])
print(f"{x = }, {y = }")

# point[0] = 5  # TypeError 💥

2
x = 2, y = 3


### `set`

Множеството пази уникални елементи, може да се променя и не гарантира реда на обхождането им. Ползваме `in` за проверка на принадлежност; достъп по индекс няма.

⚠️ `{}` е празен речник. За празно множество ползваме `set()`.

In [3]:
names = {"Иван", "Мария", "Иван"}
names.add("Петър")

print(len(names))
print("Мария" in names)
print("Ани" in names)

3
True
False


### `dict`

Речникът свързва ключове със стойности и може да се променя. Достъпваме стойност по ключ, а с `.items()` обхождаме двойките в реда на добавянето им.

Обърнете внимание, че `in` проверява **ключовете**. С `.get()` можем да зададем какво да получим при липсващ ключ:

In [4]:
grades = {"Иван": 5, "Мария": 6}
grades["Петър"] = 3

for name, grade in grades.items():
    print(f"{name}: {grade}")

print(5 in grades)
print(5 in grades.values())
print(grades.get("Ани", "няма оценка"))

Иван: 5
Мария: 6
Петър: 3
False
True
няма оценка


### Кои обекти могат да бъдат ключове?

За ключ в `dict` или елемент в `set` ни трябва обект, който може да се **хешира** (`hashable`). Хешът му трябва да остава същият през живота му, а равни обекти трябва да имат равни хешове за да могат да бъдат неразличими спрямо споментатите структури - връзката между `__hash__` и `__eq__`, която видяхме в ООП.

Числата и низовете могат да се хешират, а списъците и речниците - не. При `tuple` всичко зависи от съдържанието: **всички негови елементи също трябва да могат да се хешират**.

⚠️ Immutable и hashable не са синоними. Един `tuple` може да съдържа списък. Не можем да заменим този елемент на tuple-а, но самият списък може да се променя:

In [5]:
point = (2, 3)
labels = {point: "начална точка"}
print(labels[(2, 3)])
print(hash(point))

student = ("Мария", [5, 6])
student[1].append(4)
print(student)

# hash(student)  # TypeError: списъкът в tuple-а не може да се хешира 💥

начална точка
8409376899596376432
('Мария', [5, 6, 4])


### `str`: познатите операции върху текст

При низовете също имаме индексиране, slicing, `len` и `in`. Те са immutable: методи като `.upper()` връщат нов низ. Вече използвахме и `.split()` за разбиване на текст и `.join()` за слепване:

In [6]:
text = "Python is awesome"

print(text[0], text[-1])
print(text[:6])
print(len(text), "Python" in text)
print(text.split())
print(" | ".join(text.split()))
print(text.upper())
print(text)

P e
Python
17 True
['Python', 'is', 'awesome']
Python | is | awesome
PYTHON IS AWESOME
Python is awesome


### А ако искаме собствена колекция?

В темата за ООП видяхме как магическите методи свързват нашите обекти с познатите операции:

| Операция | Метод |
| --- | --- |
| `len(obj)` | `__len__` |
| `obj[key]` | `__getitem__` |
| `for item in obj` | `__iter__` (връща итератор) |
| `item in obj` | `__contains__` |
| `obj[key] = value` | `__setitem__` |
| `del obj[key]` | `__delitem__` |

Избираме методите според операциите, които колекцията ни ще поддържа. Например можем да позволим обхождане, индексиране и проверка на принадлежност:

In [7]:
class StudyGroup:
    def __init__(self, names):
        self.names = list(names)

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        return self.names[index]

    def __iter__(self):
        return iter(self.names)

    def __contains__(self, name):
        return name in self.names


group = StudyGroup(["Иван", "Мария"])
print(len(group), group[0])
print("Мария" in group)
print(", ".join(group))

2 Иван
True
Иван, Мария


## Още полезни операции с низове

Към познатите операции можем да добавим няколко метода за проверка на съдържанието, търсене и преброяване. Повече примери има в [документацията на `str`](https://docs.python.org/3/library/stdtypes.html#text-sequence-type-str).

In [8]:
print("самобукви".isalpha())
print("1232367457".isdigit())
print("1283912873andLetters".isalnum())
print("малкибукви".islower())
print("ГОЛЕМИБУКВИ".isupper())
print("\t\n  ".isspace())

# isdigit проверява символите; не е гаранция, че int ще приеме низа.
print("²".isdigit())  # True, но int("²") би хвърлило ValueError

True
True
True
True
True
True
True


In [9]:
filename = "lecture_05.ipynb"

print(filename.startswith("lecture_"))
print(filename.endswith(".ipynb"))
print(filename.find("05"))
print(filename.count("e"))

True
True
8
2


Библиотеката `string` предоставя някои удобни константи

In [10]:
import string

# Вместо да хардкодваме "abcdef..."
print('all letters:', string.ascii_letters)
print('all lowercase:', string.ascii_lowercase)
print('all digits as a string:', string.digits)
print('all symbols:', string.punctuation)
print('all whitespaces:', string.whitespace, string.whitespace.encode())


all letters: abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
all lowercase: abcdefghijklmnopqrstuvwxyz
all digits as a string: 0123456789
all symbols: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
all whitespaces:  	



 b' \t\n\r\x0b\x0c'


## Колко струва една операция?

Две структури могат да пазят едни и същи данни, но да ни позволяват да работим с тях с различна скорост. Затова ни интересува как расте работата, когато данните станат повече.

Ще означаваме с $n$ размера на входа — в следващите примери това е броят елементи. За низ може да ни интересува броят символи, а за две колекции може да са нужни два размера: $n$ и $m$.

С **Big O** описваме горна граница на растежа на броя операции. Например $3n + 2$ расте линейно и записваме $O(n)$: пропускаме постоянните множители и членовете, които нарастват по-бавно. Това ни помага да сравняваме подходи при голям вход.

За момента приемаме, че едно сравнение на елементи има постоянна цена. Ще броим такива стъпки, вместо да мерим секунди, които зависят и от машината.

### $O(1)$: постоянна сложност

Достъпът до елемент по индекс в списък и взимането на дължината му не изискват да обходим елементите. Работата на тези операции не расте с дължината на списъка.

Разглеждаме самия достъп, а не подготовката на данните:

In [11]:
numbers = list(range(8))  # Подготовката на този списък отнема O(n).

print(numbers[4])  # Самият достъп е O(1).
print(len(numbers))  # O(1)

4
8


### $O(n)$: линейна сложност

При търсене по стойност в списък може да се наложи да проверим всички елементи. Ако търсената стойност липсва, при два пъти повече елементи ще направим два пъти повече сравнения.

Нека преброим сравненията в обикновено последователно търсене:

In [12]:
def contains_with_count(items, target):
    comparisons = 0
    for item in items:
        comparisons += 1
        if item == target:
            return True, comparisons
    return False, comparisons


for size in (8, 16, 32):
    found, comparisons = contains_with_count(list(range(size)), -1)
    print(f"n = {size}: {found = }, {comparisons = }")

n = 8: found = False, comparisons = 8
n = 16: found = False, comparisons = 16
n = 32: found = False, comparisons = 32


### $O(\log n)$: логаритмична сложност

Ако на всяка стъпка оставяме само половината кандидати, при удвояване на входа добавяме само още една стъпка. Това е идеята на двоичното търсене в сортиран списък; пример с готовия модул `bisect` има в допълнителните материали.

Засега нека преброим колко пъти можем да намалим размера наполовина, докато остане един кандидат:

In [13]:
def halving_steps(size):
    steps = 0
    while size > 1:
        size //= 2
        steps += 1
    return steps


for size in (8, 16, 32):
    print(f"n = {size}: {halving_steps(size)} стъпки")

n = 8: 3 стъпки
n = 16: 4 стъпки
n = 32: 5 стъпки


### $O(n^2)$: квадратична сложност

Ако за всеки от $n$ елемента обхождаме отново всичките $n$ елемента, работата е $n \cdot n$. При два пъти повече елементи тя става четири пъти повече:

In [14]:
for size in (8, 16, 32):
    pairs = 0
    for left in range(size):
        for right in range(size):
            pairs += 1
    print(f"n = {size}: {pairs} наредени двойки")

n = 8: 64 наредени двойки
n = 16: 256 наредени двойки
n = 32: 1024 наредени двойки


### Няколко често срещани оценки

| Сложност | Пример | При удвояване на $n$ |
| --- | --- | --- |
| $O(1)$ | `len(items)`, `items[index]` за списък | Работата остава постоянна |
| $O(\log n)$ | Двоично търсене | Още приблизително една стъпка |
| $O(n)$ | Пълно обхождане | Приблизително два пъти повече работа |
| $O(n \log n)$ | Сортиране със `sorted` в най-лошия случай | Малко повече от два пъти |
| $O(n^2)$ | Два вложени цикъла по $n$ елемента | Приблизително четири пъти повече работа |

Това са примери за съответния растеж, а не обещание за точен брой секунди. Два алгоритъма с една и съща оценка могат да се различават по реално време и използвана памет.

### Най-лош случай, средно и амортизирано

Винаги е важно **какво точно оценяваме**:

* **Най-лош случай:** последователното търсене горе проверява $n$ елемента, ако стойността липсва. Ако е първа, стига едно сравнение. Записът $O(n)$ тук е за най-лошия случай.
* **Среден случай:** при обичайно разпределение на хешовете търсенето в `dict` и `set` е средно $O(1)$. При много колизии може да стигне $O(n)$.
* **Амортизирана оценка:** едно `list.append` понякога изисква заделяне на повече място и преместване на съществуващите елементи. За поредица от $n$ добавяния от празен списък общата работа е $O(n)$, тоест амортизирано $O(1)$ на добавяне.

Средната оценка зависи от предположения за входа или хешовете, а амортизираната разглежда общата цена на поредица от операции. Big O може да описва всяка от тези оценки — самият запис не означава непременно „най-лош случай“.

При дълги ключове или скъпо сравнение отчитаме и цената на самото хеширане или сравнение.

### А паметта?

Освен времето можем да оценим и **допълнителната памет**, извън вече подадения вход. Последователното търсене горе пази само текущия елемент и брояч: $O(1)$ допълнителна памет. Копирането на списък с `items[:]` създава място за $n$ референции: $O(n)$ допълнителна памет и $O(n)$ време.

Когато избираме структура, започваме от операциите, които ще правим най-често. Нека разгледаме познатите колекции от тази гледна точка.

## Какво стои зад `list`?

В CPython списъкът е **динамичен масив от референции към обекти**, а не свързан списък. Можем да стигнем направо до произволна позиция, но при добавяне или премахване в началото трябва да преместим останалите елементи.

| Операция | Сложност |
| --- | --- |
| `items[index]`, `len(items)` | $O(1)$ |
| `value in items` | $O(n)$ в най-лошия случай |
| `items.append(value)`, `items.pop()` | Амортизирано $O(1)$ |
| `items.insert(0, value)`, `items.pop(0)` | $O(n)$ |
| `items.clear()`, `items.reverse()` | $O(n)$ |
| `items.sort()` | $O(n \log n)$ в най-лошия случай |

При slicing цената зависи от броя копирани елементи, а при сортиране и търсене отчитаме и цената на сравненията.

In [15]:
numbers = [10, 20, 30, 40]

print(numbers.pop())  # Премахваме от края: останалите елементи запазват позициите си.
print(numbers.pop(0))  # Премахваме от началото: останалите се изместват наляво.
print(numbers)

40
10
[20, 30]


## Какво стои зад `dict`?

Речникът използва **хеш таблица**. Вместо да сравняваме ключа последователно с всички останали, хешът ни насочва къде да го търсим. Различни ключове могат да имат един и същ хеш - това е **колизия**, която структурата трябва да разреши.

Търсене, добавяне и изтриване по ключ са средно $O(1)$, а в най-лошия случай могат да достигнат $O(n)$. Пълното обхождане е $O(n)$. Тук приемаме, че хеширането и сравняването на ключовете имат постоянна цена.

### Промяна по време на обхождане

Можем да заменяме стойностите зад съществуващи ключове. Добавяне или изтриване на ключове по време на обхождане на същия речник обаче може да доведе до `RuntimeError` или до пропуснати елементи.

In [16]:
visits = {"Иван": 2, "Мария": 0, "Петър": 3}

for name in visits:
    visits[name] += 1  # Променяме стойности, ключовете остават същите.

print(visits)

{'Иван': 3, 'Мария': 1, 'Петър': 4}


In [17]:
visits = {"Иван": 2, "Мария": 0, "Петър": 3}

for name in list(visits):  # Снимка на ключовете: O(n) допълнителна памет.
    if visits[name] == 0:
        del visits[name]

print(visits)

{'Иван': 2, 'Петър': 3}


## Операции с множества

При `set` проверката за принадлежност също използва хеширане и е средно $O(1)$. Нека надградим познатите операции с обединение, сечение и разлика:

In [18]:
python_students = {"Иван", "Мария", "Петър"}
java_students = {"Мария", "Ани"}

print("Поне един от двата курса:", sorted(python_students | java_students))
print("И двата курса:", sorted(python_students & java_students))
print("Само Python:", sorted(python_students - java_students))
print("Точно един от двата курса:", sorted(python_students ^ java_students))

Поне един от двата курса: ['Ани', 'Иван', 'Мария', 'Петър']
И двата курса: ['Мария']
Само Python: ['Иван', 'Петър']
Точно един от двата курса: ['Ани', 'Иван', 'Петър']


## Копиране на вложени колекции

При **shallow copy** получаваме нов външен контейнер, но обектите вътре остават споделени. **Deep copy** копира рекурсивно и съдържанието. Разликата се вижда, когато то включва mutable обекти:

In [19]:
from copy import copy, deepcopy

original = [[1, 2], [3, 4]]
shallow = copy(original)
deep = deepcopy(original)

shallow[0].append(5)
print(f"{original = }")
print(f"{shallow = }")
print(f"{deep = }")

print(shallow is original)
print(shallow[0] is original[0])
print(deep[0] is original[0])

original = [[1, 2, 5], [3, 4]]
shallow = [[1, 2, 5], [3, 4]]
deep = [[1, 2], [3, 4]]
False
True
False


## Готови колекции за конкретни задачи

След като видяхме цената на основните операции и копирането, нека изберем по-удобни инструменти за групиране, броене и използване на множество като ключ:

### `defaultdict`

`collections.defaultdict` извиква подадена **функция без аргументи**, когато достъпим липсващ ключ с `[]`, и записва резултата като негова стойност. Например `list` е удобна фабрика, когато групираме елементи в списъци:

In [20]:
from collections import defaultdict

my_default_dict = defaultdict(list)
print(my_default_dict)
my_default_dict['what']
print('дори само с достъпване на ключа създаваме дефолтен обект', my_default_dict)

del my_default_dict['what']
for index, char in enumerate('this is a string'):
    my_default_dict[char].append(index)  # Това ще изгърми с нормален dict

my_default_dict


defaultdict(<class 'list'>, {})
дори само с достъпване на ключа създаваме дефолтен обект defaultdict(<class 'list'>, {'what': []})


defaultdict(list,
            {'t': [0, 11],
             'h': [1],
             'i': [2, 5, 13],
             's': [3, 6, 10],
             ' ': [4, 7, 9],
             'a': [8],
             'r': [12],
             'n': [14],
             'g': [15]})

### Counter

`Counter` е още един под-клас на `dict`. При инициализация може да се подаде колекция (или друг обект който може да се итерира и елементите му могат да се хешират), която бива итерирана и елементите ѝ се преброяват.
Елементите на колекцията стават ключове, а стойностите са колко пъти даден елемент е срещнат в колекцията.

In [21]:
from collections import Counter

counts = Counter("banana")
print(counts)
print(counts.most_common(2))

Counter({'a': 3, 'n': 2, 'b': 1})
[('a', 3), ('n', 2)]


### Frozenset

`frozenset` = immutable `set`. Immutability носи плюсове и минуси:
- (+) може да хешираме сета (да го използваме като ключ в `dict` или да го добавим в друг `set`)
- (-) не може да го променяме (да добавяме/премахваме елементи)

In [22]:
numbers = set([1, 2, 3])

my_dict = {
    frozenset(numbers): 'my password',
    frozenset('characters'): 'ако подадем string, ще конструираме set от буквите'
}
my_dict


{frozenset({1, 2, 3}): 'my password',
 frozenset({'a',
            'c',
            'e',
            'h',
            'r',
            's',
            't'}): 'ако подадем string, ще конструираме set от буквите'}

## Stack

`stack` (стак/стек) е LIFO (Last In First Out) структура, която ни позволява бърз достъп до последния добавен елемент. В Python няма клас `Stack`, защото `list` има всички необходими методи.

![](https://i.imgur.com/IDr4JQL.png)

In [23]:
stack = []
for x in range(100):
    stack.append(x)

first10 = 0
for i in range(10):
    first10 += stack.pop()  # Премахването от края е амортизирано O(1)

print('Сумата на първите 10 елемента е: ', first10)  # sum(100 до 90) = 945

Сумата на първите 10 елемента е:  945


## Опашка с `deque`

При опашката обработваме елементите в реда на постъпването им: **FIFO** (First In, First Out). Добавяме в единия край, а премахваме от другия.

`collections.deque` поддържа добавяне и премахване от **двата края** за $O(1)$. При `list.pop(0)` местим останалите елементи, а при `deque.popleft()` това не е нужно. Операции в средата на deque могат да отнемат $O(n)$.

![Опашка: първият добавен елемент се премахва първи](https://i.imgur.com/c8rMyYo.png)

In [24]:
from collections import deque

deq = deque(['опашка ли съм', 'ʞǝꓕɔ иѵи'])
deq.insert(1, '¿?')  # Добавянето на елемент на произволна позиция е О(n)
print(deq)

deq.append('у десно')
deq.append('у десно (1)')
deq.appendleft('у лево')

print(deq)
print('последният елемент за О(1):', deq.pop())
print('първият елемент за О(1):', deq.popleft())


deque(['опашка ли съм', '¿?', 'ʞǝꓕɔ иѵи'])
deque(['у лево', 'опашка ли съм', '¿?', 'ʞǝꓕɔ иѵи', 'у десно', 'у десно (1)'])
последният елемент за О(1): у десно (1)
първият елемент за О(1): у лево


### Пример: обхождане с опашка

Нека върховете на един граф са ключове в речник, а съседите на всеки връх - списъкът зад съответния ключ. С опашка можем да обходим графа **в ширина** (BFS): първо близките върхове, после по-далечните.

In [8]:
from collections import deque

graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': ['F'],
    'F': []
}

def bfs(graph, start):
  visited = set()
  que = deque()
  visited.add(start)
  que.append(start)

  while que:
    s = que.popleft()
    print(s, end=" ")

    for neighbor in graph[s]:
      if neighbor not in visited:
        visited.add(neighbor)
        que.append(neighbor)

bfs(graph, 'A')
print()

A B C D E F 


## Приоритетна опашка с `heapq`

При стек изваждаме последния добавен елемент, а при FIFO опашка — първия. Ако искаме всеки път да получаваме елемента с **най-малък приоритет**, използваме heap.

`heapq` поддържа такава структура върху обикновен списък. Най-малкият елемент е на индекс `0`, но целият списък **не е непременно сортиран**.

* `heapq.heapify(items)` превръща готов списък в heap за $O(n)$.
* `heapq.heappush(items, value)` добавя елемент за $O(\log n)$.
* `heapq.heappop(items)` премахва и връща най-малкия елемент за $O(\log n)$.
* `items[0]` ни дава най-малкия елемент без премахване за $O(1)$.

In [26]:
import heapq
import random

numbers = list(range(1, 11))  # [1, 2, ... 10]
random.shuffle(numbers)
print('Разбъркваме числата с random.shuffle()', numbers)

heapq.heapify(numbers)  # по-бързо от добавянето на елементи 1 по 1
print('Данните не са подредени в реда в който ги вкарваме, но 1 винаги е в началото:',
      numbers)

# Добавянето на нов по-малък елемент ще го постави в началото
heapq.heappush(numbers, 0)
print('0 винаги е в началото:', numbers)

top3 = []
for i in range(3):
    top3.append(heapq.heappop(numbers))

print('Приоритетната опашка връща елементите от най-малък към най-голям', top3)


Разбъркваме числата с random.shuffle() [1, 2, 10, 6, 9, 3, 8, 4, 7, 5]
Данните не са подредени в реда в който ги вкарваме, но 1 винаги е в началото: [1, 2, 3, 4, 5, 10, 8, 6, 7, 9]
0 винаги е в началото: [0, 1, 3, 4, 2, 10, 8, 6, 7, 9, 5]
Приоритетната опашка връща елементите от най-малък към най-голям [0, 1, 2]


## `heapq` или `PriorityQueue`?

Класът `queue.PriorityQueue` използва heap и добавя **синхронизация между нишки** и операции `put`/`get`, които могат да изчакват. Например `get()` изчаква елемент при празна опашка, а `put()` може да изчаква свободно място при зададен `maxsize`.

* **`heapq` е достатъчен**, когато решаваме алгоритмична задача или обработваме приоритети в една нишка. Работим направо със списък чрез `heappush`/`heappop`.
* **`PriorityQueue` е удобен**, когато няколко нишки подават и обработват задачи през обща опашка и ни трябват синхронизирани операции и изчакване.

И двата инструмента изваждат най-малкия приоритет първи. Тук показваме интерфейса на класа; повече за синхронизираните опашки има в допълнителните материали в края.

[Документация на `PriorityQueue`](https://docs.python.org/3.12/library/queue.html#queue.PriorityQueue).

In [27]:
from queue import PriorityQueue

jobs = PriorityQueue(maxsize=3)
jobs.put((3, "изпращане на отчет"))
jobs.put((1, "обработка на плащане"))
jobs.put((2, "генериране на фактура"))

for _ in range(3):
    print(jobs.get())

(1, 'обработка на плащане')
(2, 'генериране на фактура')
(3, 'изпращане на отчет')


## Коя структура да изберем?

Нека съберем разгледаните възможности на едно място. Започваме от задачата и операциите, които ще извършваме най-често:

| Структура | Кратък пример | Типични задачи |
| --- | --- | --- |
| `str` | `"Python is awesome"` | Текст: търсене, разделяне, форматиране и проверки на съдържанието. |
| `list` | `[5, 3, 6, 5]` | Наредени данни с повторения; достъп по индекс, промяна и сортиране на записи. |
| `tuple` | `(42.7, 23.3)` | Координати и групи от няколко стойности; ключ в речник, ако всички елементи са hashable. |
| `dict` | `{"Иван": 5, "Мария": 6}` | Търсене на запис по ключ: оценка по име, потребител по ID, индекс на стойности. |
| `set` | `{"Иван", "Мария"}` | Уникални елементи, проверки за принадлежност, общи и различни участници в две групи. |
| `frozenset` | `frozenset({"read", "write"})` | Непроменимо множество; комбинация от права или участници като ключ в речник. |
| `defaultdict(list)` | `groups[course].append(student)` | Групиране на записи по общ признак; списъци от събития за всеки потребител. |
| `Counter` | `Counter("banana")` | Честоти на букви, думи или събития; намиране на най-честите елементи. |
| Стек с `list` | `stack.append(item)` / `stack.pop()` | Undo, проверка на скоби, обхождане в дълбочина; първо обработваме последния добавен елемент. |
| `deque` | `requests.append(item)` / `requests.popleft()` | FIFO заявки и BFS; добавяне и премахване от двата края. |
| Heap с `heapq` | `heapq.heappush(heap, (1, "задача"))` | Многократен избор на най-малкия приоритет в алгоритъм или планировчик в една нишка. |
| `PriorityQueue` | `PriorityQueue(maxsize=10)` | Общи задачи между нишки, обработка по приоритет и изчакване при празна или пълна опашка. |
| `queue.Queue` | `Queue(maxsize=10)` | Общ FIFO канал между нишки, когато редът на постъпване е важен. |
| `array` | `array('d', [1.0, 2.0])` | Компактно съхранение на много числа от един и същ C тип. |

Възможностите могат да се комбинират: речник от списъци, опашка от tuple-и, множество за вече посетените върхове. Избираме според нужните операции, а после отчитаме времето и паметта им.

## Задача: изборът на структура има значение

Дадени са списък от числа и целева сума. Намерете индексите на два различни елемента с тази сума ([Two sum](https://leetcode.com/problems/two-sum/)).

Проверяването на всички двойки би отнело $O(n^2)$ време. В решението използваме речник, за да запомним индексите за всяка стойност. Получаваме средно $O(n)$ време и $O(n)$ допълнителна памет при обичайните предположения за хеширането.

In [28]:
from collections import defaultdict
def twoSum(nums, target: int):
    counter = defaultdict(list)
    for i, num in enumerate(nums):
        counter[num].append(i)

    for first in nums:
        second = target - first
        if second in counter and first != second:
            return [counter[first][0], counter[second][0]]
        elif second in counter and len(counter[first]) > 1:
            return [counter[first][0], counter[second][1]]
    return []

print(twoSum([2, 7, 11, 15], 9))
print(twoSum([3, 3], 6))

[0, 1]
[0, 1]


## Допълнителни материали и справки

Следващите секции събират още полезни инструменти от стандартната библиотека и по-сложни примери. Те могат да се използват като справка или да се разгледат при решаване на подходяща задача.


### Math

In [29]:
import math

print('Закръгляне нагоре 2.2 ->', math.ceil(2.2))
print('Закръгляне надолу 2.7 ->', math.floor(2.7))

# Може да правим побитово итериране на  числото
print('Преброяване на битовете на числото 257 =', math.ceil(math.log2(257)))
# Полезно при построяване на Trie/radix sort/заделяне на hashmap
print('Преброяване на цифрите на числото 14532 =', math.ceil(math.log10(14532)))

# Математически важни функции
print('Най-голям общ делител на 35 и 15 =', math.gcd(35, 15))
print('Можем да смятаме inverse елемента по модул 15^-1 mod 26 =', pow(15, -1, 26))

# Комбинаторика (често се среща в задачи, но има и други)
print('N choose K -> 10 choose 2 =', math.comb(10, 2))


Закръгляне нагоре 2.2 -> 3
Закръгляне надолу 2.7 -> 2
Преброяване на битовете на числото 257 = 9
Преброяване на цифрите на числото 14532 = 5
Най-голям общ делител на 35 и 15 = 5
Можем да смятаме inverse елемента по модул 15^-1 mod 26 = 7
N choose K -> 10 choose 2 = 45


`math` модулът има още много функции на които няма да се спираме, защото не се използват често:
- Тригонометрични функции
- Хиперболични функции
- Гама функция
- Конвертиране между градуси и радиани
- Други

### Array

`array` предоставя по-ефективно използване на паметта от `list`. Повече прилича на стандартните масиви в други езици. Добавянето на елемент в края на `array` е амортизирано $O(1)$.

Има същите функционалности като `list`.

При инициализиране трябва да окажем типа на елементите в списъка.

Ако се е стигнало до използване на по-оптимизирани структури в Python най-добре да използваме някоя библиотека като `numpy`.

In [30]:
%%capture
# %%capture is a jupyter instruction to hide output
from array import array

array('l')  # array<32 bit int>
array('u', 'hello \u2641')  # array<unicode character>
array('d', [1.0, 2.0, 3.14])  # array<double>
array('I', [1, 2, 3])  # array<unsigned 16 bit int>


### Двоично търсене с `bisect`

Вече използвахме `sorted(items)`, което връща нов списък, и `items.sort()`, което променя съществуващия. Когато данните са **сортирани**, можем да търсим граници чрез последователно намаляване на кандидатите наполовина: $O(\log n)$ сравнения.

Модулът `bisect` намира **позиция за вмъкване**:

* `bisect_left(items, value)` — преди всички елементи, равни на `value`.
* `bisect_right(items, value)` — след всички елементи, равни на `value`.

Получаваме позиция и когато стойността липсва. Ако искаме да проверим дали е намерена, след `index = bisect_left(items, value)` проверяваме `index < len(items)` и `items[index] == value`.

⚠️ Намирането на позиция е $O(\log n)$, но самото вмъкване в списък остава $O(n)$ заради преместването на елементи.

In [31]:
import bisect
numbers = [0, 1, 2, 4, 4, 4, 4, 7, 8, 9]
#          0  1  2  3  4  5  6  7  8  9

print('Ако елемента съществува, връща индекса на първото срещане',
      bisect.bisect_left(numbers, 4))
print('Ако елемента съществува, връща първия индекс след последното срещане',
      bisect.bisect_right(numbers, 4))

print('При липсващо 5, позицията за вмъкване отляво е',
      bisect.bisect_left(numbers, 5))
print('При липсващо 5, позицията за вмъкване отдясно е',
      bisect.bisect_right(numbers, 5))

Ако елемента съществува, връща индекса на първото срещане 3
Ако елемента съществува, връща първия индекс след последното срещане 7
При липсващо 5, позицията за вмъкване отляво е 7
При липсващо 5, позицията за вмъкване отдясно е 7


### Random

`random` модулът предоставя методи за избиране на случаен елемент, случайно число, случайни битове и тн.

Обикновено се използва за тестове (генериране на вход/изход).

За криптографски задачи (криптиране на данни, пароли, генериране на токени и тн), не трябва да се
използва `random`, а `secrets` модула. За да не навлизаме в разликите, нека да кажем че `secrets` генерира
по-рандъм изход от `random` модула :)

In [32]:
import random

# random модулът може да приеме начална стойност, която се използва
# в рандъм алгоритмите, за да върне детерминистичен изход.
# Обикновено се използва за да може даден експеримент/тест да се репродуцира
random.seed(42)

# Това е рандъм число, но заради seed = 42, винаги ще върне 82 в този notebook
random.randint(1, 100)


82

In [33]:
import random

print('рандъм float', random.random())

print('randint [0,1]:', random.randint(0, 1))
# Някои методи в random работят със затворени, някой с отворени интервали.
# Добре е да се провери преди използването им
print('randrange [0,1) не е много рандъм:', random.randrange(0, 1))
# randrange(a,b) = randint(a,b-1)

# Ако имаме колекция от елементи може да използваме choice, за да изберем произволен
# Алтернативният метод е да използваме random.randrange(0, len(collection))
print('Епа кот дойде:', random.choice(['fuck', 'marry', 'kill']))

# Може да задаваме и тежести - 50% ще е първият елемент, 25% втория и 25% третия
print('Понякога рандъм изборът не е много рандъм',
      random.choices(
          ['boiko', 'pak boiko', 'totally not boiko'],
          weights=[2, 1, 1]
      ))

# Понякога трябва да разбъркаме някяква колекция. Например преди да правим quicksort
numbers = list(range(1, 11))
random.shuffle(numbers)
print('shuffle:', numbers)

# При експерименти или когато имаме много данни искаме да изберем произволна подгрупа
print('Избор на 2 елемента без повторения:', random.sample(numbers, 2))


рандъм float 0.11133106816568039
randint [0,1]: 1
randrange [0,1) не е много рандъм: 0
Епа кот дойде: fuck
Понякога рандъм изборът не е много рандъм ['boiko']
shuffle: [8, 3, 6, 7, 1, 4, 5, 10, 9, 2]
Избор на 2 елемента без повторения: [7, 9]


In [9]:
%%capture
import random
# За пълнота трябва да споменем, че random модулът има имплементация на
# повечето видови разпределения от статистиката, но няма да навлизаме в подробности


random.uniform(1, 10)
random.gauss(1, 2)
random.expovariate(1)
random.gammavariate(10, 1)
random.normalvariate(5, 1)


### Itertools

`itertools` предоставя готови инструменти за комбиниране и обработка на поредици: слепване, натрупване на резултати, двойки и комбинации. Следващите примери показват няколко от тях; пълният списък е в [документацията](https://docs.python.org/3/library/itertools.html).

In [35]:
import itertools

# Итератор, който се връща в началото на списъка като се изчерпи.
# Наподобява Ring Buffer
cycle = itertools.cycle('ABC')
for _ in range(10):
    print(next(cycle), end=', ')
print()

# Итератор, който обединява няколко итератора/колекции
# Може да го използваме за да flatten-нем списък от списъци
for x in itertools.chain('first', ' ', 'second'):
    print(x, end=',')
print()

# Към всеки елемент от списъка се прибавя сумата на всички предишни елементи
print(list(itertools.accumulate([-20, +10, +30, -5])))
# Може да го използваме за да генерираме всички префикси на дума
print(list(itertools.accumulate('construct')))

# Генериране на всички уникални двойки. Еквивалентно на 2 вложени цикъла
print(list(itertools.combinations('ABC', 2)))

# Генериране на всички съседни двойки
print(list(itertools.pairwise([1, 5, 7, 10])))

# Всички пермутации
print(list(itertools.permutations([0, 1, 2])))

# A x B - декартово произведение
print(list(itertools.product([1, 2, 3], 'ABC')))


A, B, C, A, B, C, A, B, C, A, 
f,i,r,s,t, ,s,e,c,o,n,d,
[-20, -10, 20, 15]
['c', 'co', 'con', 'cons', 'const', 'constr', 'constru', 'construc', 'construct']
[('A', 'B'), ('A', 'C'), ('B', 'C')]
[(1, 5), (5, 7), (7, 10)]
[(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]
[(1, 'A'), (1, 'B'), (1, 'C'), (2, 'A'), (2, 'B'), (2, 'C'), (3, 'A'), (3, 'B'), (3, 'C')]


### Кеширане с `functools`

`functools` предоставя готови декоратори. Нека използваме `lru_cache`, за да запазваме резултати от вече направени изчисления и да видим как това променя работата на рекурсивната функция:

In [10]:
%%time
import functools

# Нека си дефинираме наивна рекурсивна функция
def fibonacci(n):
    if n in [0, 1]:
        return 1
    return fibonacci(n - 1) + fibonacci(n - 2)


# Броят извиквания расте експоненциално; O(2 ** n) е горна граница.
fibonacci(40)

CPU times: user 6.57 s, sys: 34.1 ms, total: 6.61 s
Wall time: 6.62 s


165580141

In [11]:
%%time
import functools

# Mоже да я забързаме ако кешираме извикванията
# Така няма да правим едно и също изчисление 2 пъти


@functools.lru_cache(maxsize=150)
def fibonacci(n):
    if n in [0, 1]:
        return 1
    return fibonacci(n - 1) + fibonacci(n - 2)


fibonacci(40)
fibonacci(150)
# Дори fib(150) което с наивната рекурсия няма да приключи
# никога минава за под 1ms


CPU times: user 79 μs, sys: 65 μs, total: 144 μs
Wall time: 147 μs


16130531424904581415797907386349

### Още един декоратор: `total_ordering`

Този пример надгражда магическите методи за сравнение от темата за ООП. Декораторът допълва останалите сравнения, когато сме задали равенство и една от наредбите:

In [38]:
# Ако имаме клас, за който искаме да имплементираме сравнение
# Може да имплементираме само __lt__ и __eq__ и останалите методи
# ще се имплементират за нас
# Защото понякога имплементирането им просто не е тривиално ...

import functools

@functools.total_ordering  # ще имплементира __gt__, __le__, и тн.
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __lt__(self, other):
        # Имплементира Z-order curve подредба на точките
        most_significant_dim = 'x'
        if self.less_msb(self.x ^ other.x, self.y ^ other.y):
            msd = 'y'

        if most_significant_dim == 'x':
            return self.x < other.x
        else:
            return self.y < other.y

    def less_msb(self, x, y):
        return x < y and x < (x ^ y)


### Синхронизирани опашки: `queue.Queue`

`queue.Queue` е FIFO опашка със синхронизация между нишки. `Queue(maxsize=42)` задава ограничен размер: добавянето може да изчака, докато се освободи място. При `maxsize=0` няма зададен лимит.

При няколко нишки `qsize()`, `empty()` и `full()` не гарантират какво ще стане при следващото `get()` или `put()`: друга нишка може вече да е променила опашката. Следващият пример използва само една нишка, за да видим основните операции.

In [40]:
import queue

que = queue.Queue()
for x in range(100):
    que.put(x)

print('Елементите ги борим с qsize: ', que.qsize())

total = 0
while not que.empty():  # Работим на 1 нишка така че това е детерминистично
    total += que.get()

print('Сумата на всички елементи е: ', total)

for x in range(100):
    que.put(x)

first10 = 0
for i in range(10):
    first10 += que.get()

print('Сумата на първите 10 елемента е: ', first10)  # sum(1 до 10) = 45


Елементите ги борим с qsize:  100
Сумата на всички елементи е:  4950
Сумата на първите 10 елемента е:  45


### CSV

Python предоставя няколко модула за работа с данни сериализирани под различен формат. Един от тези модули е за работа с CSV файлове (coma-separated values). Ще разгледаме само пример за парсене на такъв файл, но модулът има много различни функционалности и опции.

In [41]:
import csv
from io import StringIO

contents = StringIO(  # == string stream
    '''Lyubo,21,Sofia
Alex,22,Sofia
Ivan,49,"Ne Sofia"
''')

reader = csv.reader(contents)
for index, row in enumerate(reader):
    print(index, ':', row)


0 : ['Lyubo', '21', 'Sofia']
1 : ['Alex', '22', 'Sofia']
2 : ['Ivan', '49', 'Ne Sofia']


### JSON

Друг популярен формат на данни е JSON ... Разбира се в Python има модул `json`.

In [42]:
import json

my_dict = json.loads('{"foo": {"bar":["baz", null, 1.0, 2]}}')
print('От стринг конструирахме dict, който може да използваме:',
      my_dict['foo']['bar'])

print('Обратната операция на сериализиране е удобна за принтене на дебъг съобщения')
print(json.dumps(my_dict, sort_keys=True, indent=4))


От стринг конструирахме dict, който може да използваме: ['baz', None, 1.0, 2]
Обратната операция на сериализиране е удобна за принтене на дебъг съобщения
{
    "foo": {
        "bar": [
            "baz",
            null,
            1.0,
            2
        ]
    }
}


### Regex

Regex или още регулярни изрази са стрингове, които описват някакъв патърн, който може да използваме за търсене в обикновени стрингове.

`regex` може да запълни материала за цял курс, но ще се опитаме да разгледаме само често използвани примери, без да навлизаме в подробности.

Писането на регулярни изрази обикновено е трудоемка задача, за това препоръчваме инструменти като [regexr](https://regexr.com/), които изпълняват regex в реално време и имат обяснение защо даден стринг е match-нал.

In [43]:
import re

# == (.start(), .end())
print(re.search(r'123', '458971985719285712324953928572').span())

ninjas = 'ninjatreenoooinjaforestninjavoicenwvi9ew9ninja'

print('Следва ли стринга този патърн?',
      re.match(r'.*ninja$', ninjas) is not None)

print('Намери всички срещания:',
      re.findall(r'n.*?a', ninjas))

print('Разбиване на стринг на всяко срещане на патърн:',
      re.split(r'n.*?a', ninjas))

print('Субституция на патърн в стринг:',
      re.sub(r'n.*?a', '-NINJA-', ninjas))


(16, 19)
Следва ли стринга този патърн? True
Намери всички срещания: ['ninja', 'noooinja', 'ninja', 'nwvi9ew9ninja']
Разбиване на стринг на всяко срещане на патърн: ['', 'tree', 'forest', 'voice', '']
Субституция на патърн в стринг: -NINJA-tree-NINJA-forest-NINJA-voice-NINJA-


### Допълнителни задачи

- https://leetcode.com/problems/find-the-town-judge/ ([solution](https://github.com/luchev/interview-preparation/blob/master/leetcode/997.%20Find%20the%20Town%20Judge/1.py))
- https://leetcode.com/problems/number-of-islands/ ([solution](https://github.com/luchev/interview-preparation/blob/master/leetcode/200.%20Number%20of%20Islands/1.py))
- https://leetcode.com/problems/network-delay-time/ ([solution](https://github.com/luchev/interview-preparation/blob/master/leetcode/743.%20Network%20Delay%20Time/1.py))